# Three-dimensional periodic primitive-cell effective-mass reference

**Status:** Maintained calculated result for an idealized model with synthetic illustrative inputs. It is not a DFT calculation or a material-specific band structure.

**Prerequisite:** [Three-dimensional periodic primitive-cell effective-mass reference](../../../../docs/lecturenotes/solidstate/semiconductors/effmass_3d/index.md)

This laboratory applies the conventions reviewed in `effmass_2d` to the research-facing three-dimensional primitive-cell geometry.


## Model

For direct basis $A$, reciprocal basis $B=2\pi A^{-\mathsf T}$, reciprocal index $\mathbf n\in\mathbb Z^3$, and Cartesian Bloch vector $\mathbf k$,

$$\mathbf q_{\mathbf n}=\mathbf k+B\mathbf n,$$

$$E_{\mathbf n}(\mathbf k)=\frac{\hbar^2}{2}\mathbf q_{\mathbf n}^{\mathsf T}M^{-1}\mathbf q_{\mathbf n},$$

and

$$\psi_{\mathbf n\mathbf k}(\mathbf r)=\Omega^{-1/2}e^{i\mathbf q_{\mathbf n}\cdot\mathbf r}.$$

The laboratory uses angstrom, dalton, picosecond, electron-volt picosecond, and electron-volt through `UnitSystem.METAL`.


In [ ]:
import matplotlib.pyplot as plt
import numpy as np

from projectkoios.physkit.constants import SI
from projectkoios.physkit.solidstate.semiconductors.effmass_3d.cell import PeriodicPrimitiveCell3D
from projectkoios.physkit.solidstate.semiconductors.effmass_3d.model import PeriodicFreeParticle3D
from projectkoios.physkit.solidstate.semiconductors.effmass_3d.tensor import CartesianEffectiveMassTensor3D
from projectkoios.physkit.solidstate.semiconductors.effmass_3d.wavefunctions import PeriodicPlaneWave3DSampler, PeriodicPlaneWave3DSamplingRequest
from projectkoios.physkit.units import MODEL_SYSTEM_UNIT_CONVERTER, MatrixQuantity, PhysicalUnit, ScalarQuantity, UnitSystem, Unitless, VectorQuantity


## Illustrative nonorthogonal primitive cell

The basis is FCC-like only as a geometry exercise. The lattice scale and mass tensor below are synthetic inputs and are not identified as values for silicon or another material.


In [ ]:
lattice_scale = 5.4
basis = 0.5 * lattice_scale * np.array(
    [[0.0, 1.0, 1.0], [1.0, 0.0, 1.0], [1.0, 1.0, 0.0]],
    dtype=np.float64,
)
electron_mass_dalton = MODEL_SYSTEM_UNIT_CONVERTER.convert_scalar(
    ScalarQuantity(SI.me0, PhysicalUnit("kilogram")),
    UnitSystem.METAL.mass_unit,
).magnitude
mass_tensor = electron_mass_dalton * np.array(
    [[0.8, 0.08, 0.03], [0.08, 1.2, 0.05], [0.03, 0.05, 1.5]],
    dtype=np.float64,
)
cell = PeriodicPrimitiveCell3D(
    primitive_basis=MatrixQuantity(basis, UnitSystem.METAL.length_unit),
    unit_system=UnitSystem.METAL,
)
model = PeriodicFreeParticle3D(
    cell=cell,
    effective_mass=CartesianEffectiveMassTensor3D(
        tensor=MatrixQuantity(mass_tensor, UnitSystem.METAL.mass_unit),
        unit_system=UnitSystem.METAL,
    ),
)
duality = basis.T @ cell.reciprocal_basis.magnitude
assert np.allclose(duality, 2.0 * np.pi * np.eye(3), atol=1e-14)
print(f"Cell volume: {cell.volume.magnitude:.6f} Å³")
print("AᵀB =")
print(duality)


## Reciprocal modes at $\Gamma$

The calculation includes a bounded integer mode set for presentation. The model itself imposes no cutoff policy.


In [ ]:
mode_indices = tuple((i, j, k) for i in range(-2, 3) for j in range(-2, 3) for k in range(-2, 3))
spectrum = model.evaluate_modes(
    mode_indices=mode_indices,
    bloch_wave_vector=VectorQuantity(np.zeros(3), PhysicalUnit("1 / angstrom")),
)
assert spectrum.energies.magnitude[mode_indices.index((0, 0, 0))] == 0.0
fig = plt.figure(figsize=(7.2, 5.8))
ax = fig.add_subplot(projection="3d")
points = ax.scatter(*spectrum.wave_vectors.magnitude.T, c=spectrum.energies.magnitude, cmap="viridis", s=20)
ax.set(xlabel=r"$q_x$ (Å$^{-1}$)", ylabel=r"$q_y$ (Å$^{-1}$)", zlabel=r"$q_z$ (Å$^{-1}$)", title="Reciprocal modes")
fig.colorbar(points, ax=ax, label="Energy (eV)", shrink=0.7)
fig.tight_layout()


## Folded branches on an illustrative reciprocal-coordinate path

The path tests three-dimensional reciprocal-coordinate handling. It is not labeled as a crystallographic high-symmetry path.


In [ ]:
vertices = np.array([[0.0, 0.0, 0.0], [0.5, 0.0, 0.0], [0.5, 0.5, 0.0], [0.5, 0.5, 0.5], [0.0, 0.0, 0.0]])
segments = []
for start, stop in zip(vertices[:-1], vertices[1:], strict=True):
    t = np.linspace(0.0, 1.0, 35, endpoint=False)
    segments.append(start[None, :] + t[:, None] * (stop - start)[None, :])
fractional_k = np.vstack((*segments, vertices[-1:]))
cartesian_k = fractional_k @ cell.reciprocal_basis.magnitude.T
small_modes = tuple((i, j, k) for i in range(-1, 2) for j in range(-1, 2) for k in range(-1, 2))
branches = []
for k_vector in cartesian_k:
    energies = model.evaluate_modes(
        mode_indices=small_modes,
        bloch_wave_vector=VectorQuantity(k_vector, PhysicalUnit("1 / angstrom")),
    ).energies.magnitude
    branches.append(np.sort(energies)[:8])
fig, ax = plt.subplots(figsize=(7.2, 4.6))
ax.plot(np.asarray(branches))
ax.set(xlabel="Path sample", ylabel="Energy (eV)", title="Eight lowest folded branches")
ax.grid(alpha=0.25)
fig.tight_layout()


## Plane-wave slice and volume normalization

A plane wave has constant $|\psi|^2$. A two-dimensional fractional-coordinate slice exposes its phase without replacing the three-dimensional normalization check.


In [ ]:
selected = model.evaluate_modes(
    mode_indices=((1, -1, 1),),
    bloch_wave_vector=VectorQuantity(np.array([0.06, -0.04, 0.03]), PhysicalUnit("1 / angstrom")),
)
axis = np.arange(70, dtype=np.float64) / 70.0
u, v = np.meshgrid(axis, axis, indexing="ij")
w = np.full_like(u, 0.25)
points = np.column_stack((u.ravel(), v.ravel(), w.ravel()))
sampling = PeriodicPlaneWave3DSampler().action(
    request=PeriodicPlaneWave3DSamplingRequest(
        spectrum=selected,
        fractional_coordinates=MatrixQuantity(points, Unitless()),
    )
)
psi = sampling.amplitudes.magnitude[:, 0].reshape(u.shape)
represented_norm = cell.volume.magnitude * float(np.abs(psi[0, 0]) ** 2)
assert np.isclose(represented_norm, 1.0, atol=1e-14)
fig, axes = plt.subplots(1, 2, figsize=(10.8, 4.2), constrained_layout=True)
phase = axes[0].pcolormesh(u, v, psi.real, shading="auto", cmap="coolwarm")
axes[0].set(title=r"$\mathrm{Re}\,\psi$ at $\xi_3=0.25$", xlabel=r"$\xi_1$", ylabel=r"$\xi_2$")
fig.colorbar(phase, ax=axes[0])
density = axes[1].pcolormesh(u, v, np.abs(psi) ** 2, shading="auto", cmap="magma")
axes[1].set(title=r"$|\psi|^2$", xlabel=r"$\xi_1$", ylabel=r"$\xi_2$")
fig.colorbar(density, ax=axes[1], label=r"Å$^{-3}$")


## Interpretation and limits

The calculated results verify the documented idealized reciprocal-space model under synthetic inputs. They establish direct/reciprocal duality, dimensional units, exact mode construction, and volume normalization. They do not establish semiconductor parameters, parent-model fidelity, DFT agreement, or scientific validation for silicon.

## Exercises

1. Rotate $A$, $M$, and $\mathbf k$ together and verify energy invariance.
2. Compare isotropic and anisotropic tensors while retaining the same reciprocal modes.
3. Explain why a primitive-cell basis change requires a corresponding transformation of integer mode labels.
4. Design an independent periodic real-space operator without conflating its discretization error with effective-mass model error.
